# Two point processes coupled through delayed alpha kernels

**Showcases:** point processes (`.point_process`) whose intensities depend on a filtered input that arrives after a transmission delay. Two units excite each other; unit $i$ fires with intensity $a\,e^{v_i}$ and

$$\tau\,\dot v_i = -(v_i - E) + \sum_j w_{ij}\,(g * n_j)(t), \qquad g(t) = \frac{t-d}{\tau_g^2}\,e^{-(t-d)/\tau_g}\,\Theta(t-d),$$

an alpha kernel delayed by $d$ ($g$ = `Kernel.delay('d') * Kernel.alpha('taug')`). We compute the spike cross-covariance $C_{12}(\tau) = \mathrm{Cov}(n_1(t), n_2(t+\tau))$ at tree level and one loop and compare with a direct simulation. The delay moves the peaks of $C_{12}$ to $|\tau| \approx d + \tau_g$.

Run time: about five minutes (theory ≈ 1.5 min, simulation ≈ 2–3 min).

In [ ]:
%matplotlib inline
try:
    preparser(False)   # Sage kernel: plain Python syntax and numbers in the cells below
except NameError:
    pass
import os, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
# locate the repository root (the folder holding the 'api' package) from any working directory
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'api')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(_root)
def ells(r):   # C(tau) per loop order (real parts), a list indexed by the loop order
    return [np.real(np.asarray(r['C_tau_by_ell'][l])) for l in sorted(r['C_tau_by_ell'])]
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
from api.sde import SDE
from api.noise import Gaussian
from api.delay import Kernel
from api import compute_cumulants

def delay_report(r):
    d = r.get('delay') or {}
    return {k: d.get(k) for k in ('tree', 'approximant', 'n', 'orders_run', 'target_met') if k in d}

In [ ]:
P = dict(a=1.0, E=-1.5, tau=2.0, taug=1.0, d=1.5, w=0.5)
hk = (SDE('delayed-alpha Hawkes pair')
      .population('U', size=2)
      .parameter('a', default=P['a'], domain='positive').parameter('E', default=P['E'])
      .parameter('tau', default=P['tau'], domain='positive').parameter('taug', default=P['taug'], domain='positive')
      .parameter('d', default=P['d'], domain='positive')
      .parameter('w', default=[[0.0, P['w']], [P['w'], 0.0]], indexed_by=['U', 'U'])
      .define_kernel('g', kernel=Kernel.delay('d') * Kernel.alpha('taug'))
      .field('v', population='U')
      .point_process('n', 'a*exp(v)', population='U')
      .equation('v', lhs='(tau*Dt + 1)*v', rhs='E + sum(w[i,j]*Conv(g, n[j]) for j in U)'))
hk.show()
tau = np.linspace(-6, 6, 25)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    res = compute_cumulants(hk.build(), k=2, max_ell=1, external_fields=[('n', 1), ('n', 2)],
                            tau_grid=tau, parallel=False, verbose=False)
print(f"{time.time() - t0:.0f} s; delay:", delay_report(res))
print('mean-field rates:', [float(x) for x in np.atleast_1d(res['mf_values']['nstar'])])
C = ells(res)

## Independent simulation

From scratch, not the pipeline: per time step each unit spikes with probability $a e^{v_i}\mathrm dt$; spikes enter a ring buffer and arrive $d$ later at the other unit's two-stage exponential filter (the alpha kernel); $v$ relaxes exactly between steps. Many independent chains, spike counts binned at 0.25.

In [ ]:
from numba import njit

@njit(cache=False)
def run_chain(nsteps, burn, seed, dt, a, E, tau, taug, d, w, BIN, out_counts):
    np.random.seed(seed)
    ev, eg = np.exp(-dt / tau), np.exp(-dt / taug)
    v = np.array([E, E]); s1 = np.zeros(2); s2 = np.zeros(2)
    D = int(round(d / dt)); buf = np.zeros((D, 2), dtype=np.int64); bp = 0
    nb = out_counts.shape[0]
    for k in range(burn + nsteps):
        sp0 = 1 if np.random.random() < a * np.exp(v[0]) * dt else 0
        sp1 = 1 if np.random.random() < a * np.exp(v[1]) * dt else 0
        arr0 = buf[bp, 0]; arr1 = buf[bp, 1]
        buf[bp, 0] = sp0; buf[bp, 1] = sp1; bp = (bp + 1) % D
        in0 = arr1 / taug; in1 = arr0 / taug                  # unit 1 drives unit 0 and vice versa
        s2[0] = s2[0] * eg + s1[0] * (dt / taug) * eg; s1[0] = s1[0] * eg + in0
        s2[1] = s2[1] * eg + s1[1] * (dt / taug) * eg; s1[1] = s1[1] * eg + in1
        v[0] = E + (v[0] - E) * ev + (w * s2[0]) * (1 - ev)
        v[1] = E + (v[1] - E) * ev + (w * s2[1]) * (1 - ev)
        if k >= burn:
            b = (k - burn) // BIN
            if b < nb:
                out_counts[b, 0] += sp0; out_counts[b, 1] += sp1
    return 0

dt, BIN, L, chains = 0.005, 50, 1.0e5, 200          # bin = 0.25; 2e7 time units in total
Db = dt * BIN; nsteps = int(L / dt); nb = nsteps // BIN
lags = np.arange(-24, 25, 2)                        # tau = -6 .. 6 in steps of 0.5
per_chain, rates = [], []
t0 = time.time()
for ch in range(chains):
    cnt = np.zeros((nb, 2), dtype=np.int64)
    run_chain(nsteps, int(200 / dt), 1000 + ch, dt, P['a'], P['E'], P['tau'], P['taug'], P['d'], P['w'], BIN, cnt)
    x0 = cnt[:, 0] - cnt[:, 0].mean(); x1 = cnt[:, 1] - cnt[:, 1].mean()
    row = [np.dot(x0[:nb - l], x1[l:]) / (nb - l) if l >= 0 else np.dot(x0[-l:], x1[:nb + l]) / (nb + l) for l in lags]
    per_chain.append(np.array(row) / Db ** 2); rates.append(cnt.mean(0) / Db)
per_chain = np.array(per_chain)
cov, se = per_chain.mean(0), per_chain.std(0, ddof=1) / np.sqrt(chains)
print(f"simulation {time.time() - t0:.0f} s; rates {np.mean(rates, 0)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tau, C[0], 'k--', label='tree')
ax.plot(tau, C[0] + C[1], 'C0', label='tree + 1 loop')
ax.errorbar(lags * Db, cov, yerr=2 * se, fmt='o', ms=3, color='C3', label='simulation (±2 SE)')
ax.set_xlabel(r'$\tau$'); ax.set_ylabel(r'$C_{12}(\tau)$'); ax.legend(); plt.show()
z = (cov - (C[0] + C[1])) / se
print(f"tree + 1 loop vs simulation: max |z| = {np.max(np.abs(z)):.1f} over {len(z)} lags; "
      f"tree alone: max |z| = {np.max(np.abs((cov - C[0]) / se)):.1f}")

The delay shifts the two peaks of the cross-covariance out to $|\tau| \approx d + \tau_g$. The tree level already captures the shape; the one-loop correction is small at this coupling and comparable to the simulation error. The simulation carries an $O(\mathrm dt)$ bias of order one percent from drawing at most one spike per step.